# Notebook 03 — Build Model & Verify Forward Pass
**Run after:** `02_split_and_dataloader.ipynb`

This notebook:
1. Builds the full EfficientNet-B1 + MC Dropout + Evidential Head model
2. Prints exact parameter counts and which layers are frozen
3. Runs one forward pass and shows all output shapes
4. Runs MC Dropout inference (20 passes) and shows uncertainty output
5. Demonstrates backbone freeze / unfreeze for two-stage training


In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
import warnings; warnings.filterwarnings('ignore')
import torch
import torch.nn.functional as F
import numpy as np

from models.architecture import (
    RobustMedicalClassifier, model_summary,
    compute_uncertainty_from_dirichlet, NUM_CLASSES
)
from data.dataset import IDX_TO_CLASS
from models.trainer import get_device

device = get_device()
print(f"Device: {device}")


  Device: Apple M2 GPU (MPS) ✅ — training will use GPU cores
Device: mps


## Build the Model (Stage 1 — Frozen Backbone)
First time this runs, it downloads EfficientNet-B1 ImageNet weights (~30MB).
After that, it loads from cache instantly.


In [2]:
import time
t0    = time.time()
model = RobustMedicalClassifier(num_classes=NUM_CLASSES, freeze_backbone=True)
model = model.to(device)
print(f"Model built in {time.time()-t0:.1f}s")


  Backbone FROZEN (Stage 1 training)
Model built in 0.3s


## Parameter Breakdown

In [3]:
model_summary(model)

print("\nDetailed breakdown:")
print(f"{'Module':<15} {'Total':>12} {'Trainable':>12} {'Status'}")
print(f"{'─'*15} {'─'*12} {'─'*12} {'─'*20}")
for name, module in model.named_children():
    total     = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    status    = '🔒 frozen' if trainable == 0 else f'✅ trainable'
    print(f"{name:<15} {total:>12,} {trainable:>12,} {status}")



MODEL SUMMARY
  Total parameters:      7,305,006
  Trainable:               791,822
  Frozen (backbone):     6,513,184

  Architecture:
    Backbone:  EfficientNet-B1 (ImageNet pretrained)
    Head:      Linear(1280→512) → MCDrop(0.4) → Linear(512→256) → MCDrop(0.3)
    Output 1:  Standard classifier → 7 class logits
    Output 2:  Evidential head → 7 Dirichlet α parameters
    MC passes: 20 at inference time

Detailed breakdown:
Module                 Total    Trainable Status
─────────────── ──────────── ──────────── ────────────────────
backbone           6,513,184            0 🔒 frozen
head                 788,224      788,224 ✅ trainable
classifier             1,799        1,799 ✅ trainable
evidential             1,799        1,799 ✅ trainable


## One Forward Pass
Let's run a fake batch through and inspect every output tensor.


In [4]:
fake_batch = torch.randn(4, 3, 224, 224).to(device)

t0     = time.time()
output = model(fake_batch)
ms     = (time.time() - t0) * 1000

print(f"Forward pass: {ms:.0f}ms for 4 images ({ms/4:.0f}ms/image)")
print()
print(f"output['logits']   shape: {output['logits'].shape}  ← raw class scores")
print(f"output['alpha']    shape: {output['alpha'].shape}   ← Dirichlet params (all > 0)")
print(f"output['features'] shape: {output['features'].shape} ← for Grad-CAM")
print()
probs = F.softmax(output['logits'], dim=1)
print(f"Softmax probs (image 0): {probs[0].detach().cpu().numpy().round(3)}")
print(f"Dirichlet alpha  (img 0): {output['alpha'][0].detach().cpu().numpy().round(3)}")
print(f"Alpha sum (total evidence): {output['alpha'][0].sum().item():.2f}")


Forward pass: 926ms for 4 images (232ms/image)

output['logits']   shape: torch.Size([4, 7])  ← raw class scores
output['alpha']    shape: torch.Size([4, 7])   ← Dirichlet params (all > 0)
output['features'] shape: torch.Size([4, 256]) ← for Grad-CAM

Softmax probs (image 0): [0.168 0.093 0.137 0.036 0.093 0.403 0.07 ]
Dirichlet alpha  (img 0): [1.523 1.878 1.62  1.736 1.666 2.08  1.728]
Alpha sum (total evidence): 12.23


## Understanding the Evidential Output
The `alpha` values are Dirichlet parameters. Here's what they mean:
- High alpha for one class → model has strong evidence for that class
- All alphas near 1 → model has almost no evidence → high uncertainty
- Sum of alphas = total evidence accumulated


In [5]:
unc = compute_uncertainty_from_dirichlet(output['alpha'].detach())

print(f"{'':5} {'Pred':>8} {'Confidence':>12} {'Aleatoric':>12} {'Epistemic':>12}")
print(f"{'─'*5} {'─'*8} {'─'*12} {'─'*12} {'─'*12}")
for i in range(4):
    pred = IDX_TO_CLASS[unc['probs'][i].argmax().item()]
    conf = unc['probs'][i].max().item()
    aleat = unc['aleatoric'][i].item()
    epist = unc['epistemic'][i].item()
    print(f"Img {i+1}: {pred:>8} {conf:>12.3f} {aleat:>12.3f} {epist:>12.3f}")

print("\nNote: values are random (untrained model) — will be meaningful after training")


          Pred   Confidence    Aleatoric    Epistemic
───── ──────── ──────────── ──────────── ────────────
Img 1:     vasc        0.170        1.941        0.572
Img 2:      mel        0.186        1.933        0.577
Img 3:     vasc        0.156        1.943        0.570
Img 4:     vasc        0.225        1.920        0.577

Note: values are random (untrained model) — will be meaningful after training


## MC Dropout Inference
This is what runs during evaluation to estimate uncertainty.
We keep dropout ACTIVE and run 20 forward passes.
The **variance** across passes = epistemic uncertainty.


In [6]:
single_img = torch.randn(1, 3, 224, 224).to(device)

t0     = time.time()
result = model.predict_with_uncertainty(single_img, n_passes=20)
ms     = (time.time() - t0) * 1000

print(f"20-pass MC Dropout: {ms:.0f}ms")
print()
print(f"mean_probs:      {result['mean_probs'][0].cpu().numpy().round(3)}")
print(f"predicted_class: {IDX_TO_CLASS[result['predicted_class'][0].item()]}")
print(f"confidence:      {result['confidence'][0].item():.3f}")
print(f"mc_uncertainty:  {result['mc_uncertainty'][0].item():.4f}  ← variance across 20 passes")
print(f"ev_aleatoric:    {result['ev_aleatoric'][0].item():.4f}  ← data noise")
print(f"ev_epistemic:    {result['ev_epistemic'][0].item():.4f}  ← model ignorance")


20-pass MC Dropout: 1528ms

mean_probs:      [0.174 0.124 0.117 0.155 0.15  0.127 0.152]
predicted_class: nv
confidence:      0.174
mc_uncertainty:  0.0002  ← variance across 20 passes
ev_aleatoric:    1.9455  ← data noise
ev_epistemic:    0.5926  ← model ignorance


## Two-Stage Training: Freeze vs Unfreeze

In [7]:
before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stage 1 (frozen backbone):   {before:>10,} trainable params")

model.unfreeze_backbone()
after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stage 2 (unfrozen backbone): {after:>10,} trainable params")
print(f"Newly unlocked:              {after-before:>10,} backbone params")
print()
print("In Stage 2, backbone gets 10× smaller LR (5e-5 vs 5e-4)")
print("This prevents catastrophic forgetting of ImageNet features.")
print("\n→ NEXT: Open notebook 04_loss_functions.ipynb")


Stage 1 (frozen backbone):      791,822 trainable params
  Backbone UNFROZEN — use 0.1× LR for backbone params
Stage 2 (unfrozen backbone):  7,305,006 trainable params
Newly unlocked:               6,513,184 backbone params

In Stage 2, backbone gets 10× smaller LR (5e-5 vs 5e-4)
This prevents catastrophic forgetting of ImageNet features.

→ NEXT: Open notebook 04_loss_functions.ipynb
